In [ ]:
import requests
import pandas as pd
from time import sleep
import csv

API_KEY = "709aac7a-fe74-4506-94da-b463c55192ab"
SEARCH_TERMS = ["abortion"]

def search_guardian(query, pages=10):
    
    url = "https://content.guardianapis.com/search"
    articles = []

    for page in range(1, pages + 1):
        params = {
            "q": query,
            "edition": "us",
            "page": page,
            "page-size": 50,
            "show-fields": "headline,bodyText",
            "api-key": API_KEY
        }

        res = requests.get(url, params=params).json()
        results = res.get("response", {}).get("results", [])

        if not results:
            break

        for item in results:
            fields = item.get("fields", {})
            articles.append({
                "keyword": query,
                "title": fields.get("headline", ""),
                "text": fields.get("bodyText", ""),
                "date": item.get("webPublicationDate", ""),
                "url": item.get("webUrl", "")
            })

        sleep(0.5) 

    return articles


all_data = []
for term in SEARCH_TERMS:
    data = search_guardian(term)
    all_data.extend(data)

df = pd.DataFrame(all_data)
df.to_csv(
    "guardian_abortion_articles_us.csv",
    index=False,
    encoding="utf-8-sig",
    quoting=csv.QUOTE_ALL
)


